# Evaluating Class Superposition in CNN Embedding Space

**Objective:** To quantitatively evaluate the superposition (overlap) between samples of different classes in the embedding space of a CNN and to investigate if the degree of superposition is related to the model's classification performance.

**Methodology:**
1.  Train two CNN models on the CIFAR-10 dataset:
    * A **"Good Model"** trained for a sufficient number of epochs to achieve high accuracy.
    * A **"Poor Model"** trained for only a few epochs to achieve low accuracy.
2.  Extract the embeddings (feature vectors) from the penultimate layer of both models for the test dataset.
3.  Quantify the class separability using the ratio of inter-class distance to intra-class distance.
4.  Visualize the embedding spaces using t-SNE to qualitatively assess the class overlap.

In [ ]:
# Import necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.metrics import pairwise_distances
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Define transformations for the data
# For training, we'll use data augmentation; for testing, just normalization.
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

# Load CIFAR-10 dataset
try:
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
except Exception as e:
    print(f"Failed to download CIFAR-10. You might be offline. Attempting to load from local './data' directory.")
    print("Please ensure the data is available locally if the download fails.")
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=False, transform=transform_train)
    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=False, transform=transform_test)


trainloader = DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
testloader = DataLoader(testset, batch_size=128, shuffle=False, num_workers=2)

# Define class names for CIFAR-10
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

In [ ]:
# A simple CNN for CIFAR-10 classification
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        # Convolutional layers
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        # Flatten the feature maps
        self.flatten = nn.Flatten()
        
        # This is our embedding layer
        self.embedding = nn.Linear(128 * 4 * 4, 256) 
        
        # Classifier layer
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.flatten(x)
        
        # Get embeddings
        embeddings = self.embedding(x)
        embeddings = nn.ReLU()(embeddings)
        
        # Classify
        output = self.classifier(embeddings)
        return output

    def get_embeddings(self, x):
        """Helper function to extract embeddings."""
        x = self.conv_layers(x)
        x = self.flatten(x)
        embeddings = self.embedding(x)
        embeddings = nn.ReLU()(embeddings)
        return embeddings

In [ ]:
# Function to train the model
def train_model(model, trainloader, criterion, optimizer, epochs):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        progress_bar = tqdm(enumerate(trainloader), total=len(trainloader), desc=f"Epoch {epoch+1}/{epochs}")
        for i, data in progress_bar:
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            progress_bar.set_postfix({'loss': running_loss / (i + 1)})
    print("Finished Training")

# Function to evaluate the model
def evaluate_model(model, testloader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in testloader:
            images, labels = data
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = 100 * correct / total
    print(f'Accuracy of the network on the 10000 test images: {accuracy:.2f} %')
    return accuracy

In [ ]:
# --- Train the "Good Model" ---
print("--- Training the Good Model ---")
good_model = SimpleCNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(good_model.parameters(), lr=0.001)

# Train for more epochs to get good performance
train_model(good_model, trainloader, criterion, optimizer, epochs=25)
good_model_accuracy = evaluate_model(good_model, testloader)


# --- Train the "Poor Model" ---
print("\n--- Training the Poor Model ---")
poor_model = SimpleCNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(poor_model.parameters(), lr=0.001)

# Train for very few epochs to get poor performance
train_model(poor_model, trainloader, criterion, optimizer, epochs=2)
poor_model_accuracy = evaluate_model(poor_model, testloader)

In [ ]:
def get_all_embeddings(model, dataloader):
    """Extracts embeddings and labels for the entire dataset."""
    model.eval()
    all_embeddings = []
    all_labels = []
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            embeddings = model.get_embeddings(images)
            all_embeddings.append(embeddings.cpu().numpy())
            all_labels.append(labels.numpy())
            
    return np.concatenate(all_embeddings), np.concatenate(all_labels)

# Extract embeddings for both models
print("Extracting embeddings for the Good Model...")
good_embeddings, good_labels = get_all_embeddings(good_model, testloader)

print("Extracting embeddings for the Poor Model...")
poor_embeddings, poor_labels = get_all_embeddings(poor_model, testloader)

print(f"Shape of embeddings: {good_embeddings.shape}")

### Quantitative Metric: Inter-class vs. Intra-class Distance

To quantify class separability, we will compute the following:
-   **Intra-class distance**: The average distance between samples *within the same class*. A smaller value means samples of the same class are clustered tightly.
-   **Inter-class distance**: The average distance between samples from *different classes*. A larger value means different classes are far apart.

We will then calculate the **Separability Ratio = (Average Inter-class Distance) / (Average Intra-class Distance)**. A higher ratio indicates better-separated classes and less superposition.

In [ ]:
def calculate_separability_ratio(embeddings, labels):
    """
    Calculates the ratio of average inter-class distance to average intra-class distance.
    """
    num_classes = len(np.unique(labels))
    total_intra_dist = 0
    total_inter_dist = 0
    
    # Calculate pairwise distance matrix
    dist_matrix = pairwise_distances(embeddings, metric='euclidean')
    
    # --- Calculate Intra-class distances ---
    intra_count = 0
    for i in range(num_classes):
        class_mask = (labels == i)
        class_indices = np.where(class_mask)[0]
        if len(class_indices) < 2:
            continue
        
        # Get distances only for pairs within this class
        class_dist_matrix = dist_matrix[class_indices][:, class_indices]
        # Sum the upper triangle of the distance matrix (to avoid double counting)
        intra_dist = np.sum(np.triu(class_dist_matrix, k=1))
        
        num_pairs = len(class_indices) * (len(class_indices) - 1) / 2
        total_intra_dist += intra_dist
        intra_count += num_pairs

    avg_intra_dist = total_intra_dist / intra_count if intra_count > 0 else 0

    # --- Calculate Inter-class distances ---
    # We can calculate this more efficiently by taking the total pair distance
    # and subtracting the total intra-class distance.
    total_dist = np.sum(np.triu(dist_matrix, k=1))
    total_inter_dist = total_dist - total_intra_dist
    
    total_pairs = len(labels) * (len(labels) - 1) / 2
    inter_count = total_pairs - intra_count
    
    avg_inter_dist = total_inter_dist / inter_count if inter_count > 0 else 0
    
    # --- Calculate Ratio ---
    separability_ratio = avg_inter_dist / avg_intra_dist if avg_intra_dist > 0 else float('inf')
    
    return avg_intra_dist, avg_inter_dist, separability_ratio

# --- Analyze the "Good Model" ---
good_avg_intra, good_avg_inter, good_ratio = calculate_separability_ratio(good_embeddings, good_labels)
print("--- Good Model Analysis ---")
print(f"Model Accuracy: {good_model_accuracy:.2f}%")
print(f"Average Intra-class Distance: {good_avg_intra:.4f}")
print(f"Average Inter-class Distance: {good_avg_inter:.4f}")
print(f"Separability Ratio: {good_ratio:.4f}")

print("-" * 30)

# --- Analyze the "Poor Model" ---
poor_avg_intra, poor_avg_inter, poor_ratio = calculate_separability_ratio(poor_embeddings, poor_labels)
print("--- Poor Model Analysis ---")
print(f"Model Accuracy: {poor_model_accuracy:.2f}%")
print(f"Average Intra-class Distance: {poor_avg_intra:.4f}")
print(f"Average Inter-class Distance: {poor_avg_inter:.4f}")
print(f"Separability Ratio: {poor_ratio:.4f}")

In [ ]:
def plot_tsne(embeddings, labels, title):
    """
    Performs t-SNE dimensionality reduction and plots the result.
    To speed up, we'll use a subset of the data for plotting.
    """
    print(f"Running t-SNE for '{title}'...")
    
    # Use a subset for faster t-SNE
    subset_size = 2000
    indices = np.random.choice(embeddings.shape[0], subset_size, replace=False)
    subset_embeddings = embeddings[indices]
    subset_labels = labels[indices]
    
    tsne = TSNE(n_components=2, verbose=1, perplexity=40, n_iter=300)
    tsne_results = tsne.fit_transform(subset_embeddings)
    
    plt.figure(figsize=(12, 10))
    sns.scatterplot(
        x=tsne_results[:, 0], y=tsne_results[:, 1],
        hue=subset_labels,
        palette=sns.color_palette("hls", 10),
        legend="full"
    )
    plt.title(title, fontsize=16)
    plt.xlabel("t-SNE Component 1")
    plt.ylabel("t-SNE Component 2")
    # Place legend outside the plot
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0., labels=classes)
    plt.show()

# Visualize the embeddings
plot_tsne(good_embeddings, good_labels, f't-SNE of Good Model Embeddings (Accuracy: {good_model_accuracy:.2f}%)')
plot_tsne(poor_embeddings, poor_labels, f't-SNE of Poor Model Embeddings (Accuracy: {poor_model_accuracy:.2f}%)')

## Conclusion and Analysis
